# 02 Z-stack dataset: CNN, chip network and feature control

Grouped cross-validation within the z-stack dataset (static cells, Andor Zyla, resampled to 0.65 µm/px).

* **Split by stack:** all focal planes of one stack are in the same fold (five folds).
* **Split by acquisition day:** all images of one day are in the same fold. Each day contains only one cell line,
  and only two NIH3T3 days exist, so this split has two folds.

In both splits, 20 % of the training stacks are used for validation (epoch selection by macro-F1).
Models: CNN (64 × 64), chip network (8 × 8), three seeds each, and the feature control (logistic regression on
crop height, crop width, fill fraction and raw contrast).

Thesis: Results, Section "Classification of the Andor Zyla z-stack dataset" (Tables `tab:res-zstack` and
`tab:res-perday`). The confusion matrices of the appendix are drawn in notebook 08.

Output: `results/02_zstack_metrics.csv`, `_predictions.csv.gz`, `_history.csv`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

## 1. Data

In [ ]:
Z = C.load_data(sets=['zstack'])['zstack']
SPLITS = {'zstack_cv_date': 'group_date', 'zstack_cv': 'group_stack'}   # scenario: group column
for scenario, group_col in SPLITS.items():
    print(f'{scenario:15s} groups per class:', Z.meta.groupby('label')[group_col].nunique().to_dict())

## 2. Cross-validation
Validation always uses whole stacks of the training part (`val_group_col='group_stack'`). With only two NIH3T3 days,
the split by day falls back to two folds (a warning is printed).

In [ ]:
metrics, preds, hist = [], [], []
for kind, size in C.MODELS:
    for scenario, group_col in SPLITS.items():
        m, p, h = M.run_grouped_cv(Z.meta, Z.X, kind, size, group_col, C.TRAIN, C.N_FOLDS,
                                   C.SEEDS_BY_MODEL[kind], C.VAL_FRAC, scenario=scenario,
                                   val_group_col='group_stack')
        metrics.append(m); preds.append(p); hist.append(h)
for scenario, group_col in SPLITS.items():
    metrics.append(M.run_control_cv(Z.meta, Z.feat, group_col, C.N_FOLDS, scenario=scenario))
metrics = pd.concat(metrics, ignore_index=True)
preds = pd.concat(preds, ignore_index=True)
hist = pd.concat(hist, ignore_index=True)
M.save_results('02_zstack', C.RESULTS_DIR, metrics, preds, hist)

## 3. Results
Macro-F1, mean ± standard deviation over folds and seeds (feature control: over folds).

In [ ]:
display(metrics.groupby(['scenario', 'model']).macro_f1.agg(['mean', 'std', 'count']).round(3))

Fraction of correctly classified images per class, from the predictions of all folds and seeds (the same values as
the percentages in the confusion matrices of the appendix).

In [ ]:
recall = (preds.groupby(['scenario', 'model', 'label']).correct.mean().unstack('label').round(3))
display(recall)

### Result per held-out acquisition day
Split by day only. Every day contains one cell line, so the fraction of correctly classified images equals the
recall of that class. `n` counts distinct test images, the fraction uses the predictions of all seeds.

In [ ]:
day_preds = preds[preds.scenario.eq('zstack_cv_date')]
per_day = day_preds.groupby(['label', 'date', 'model']).correct.mean().unstack('model')
per_day.insert(0, 'n', day_preds.groupby(['label', 'date']).uid.nunique())
per_day.insert(0, 'binning', [{0.325: '1x1', 0.65: '2x2'}[Z.meta.loc[Z.meta.date.eq(d), 'px_um'].iloc[0]]
                              for d in per_day.index.get_level_values('date')])
display(per_day.round(3))